In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Steps I am going to do:
Step 1: Setup
Step 2: EDA
Step 3: Baseline Model
Step 4: Submit Baseline
Step 5: Feature Engineering/Preprocessing
Step 6: Embedding-based model
Step 7: Threshold tuning
Step 8: Iterate


Step 1: Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/quora-insincere-questions-classification/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/quora-insincere-questions-classification/test.csv')


Step 2: EDA


In [ ]:
# Create word count feature
train_df['word_count'] = train_df['question_text'].apply(lambda x: len(str(x).split()))


In [ ]:
from sklearn.model_selection import train_test_split

# features and target split
X = train_df['question_text']
y = train_df['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

print(f"Train set shape: {X_train.shape} | Insincere count: {y_train.sum()} ({y_train.mean():.2%})")
print(f"Val set shape:   {X_val.shape}  | Insincere count: {y_val.sum()} ({y_val.mean():.2%})")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        # Captures single words and 2-word phrases
    min_df=5,                  # Drops ultra-rare tokens and typos
    max_features=50000,        # Keeps the vocab size manageable
    stop_words='english'       # Removes starndard noise words
)

# fit on training data and transform both sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"Vocab size: {len(vectorizer.vocabulary_)}")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

y_val_proba = model.predict_proba(X_val_tfidf)[:, 1]


In [ ]:
from sklearn.metrics import f1_score, classification_report

def find_best_threshold(y_true, y_proba):
    """
    Sweeps thresholds from 0.01 to 0.99 to find the cutoff that maximizes the F1-score.
    Returns the best threshold and its corresponding score.
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_thresh = 0.5
    best_f1 = 0.0

    for thresh in thresholds:
        # Convert probabilities to binary predictions based on the current threshold
        y_pred = (y_proba >= thresh).astype(int)
        score = f1_score(y_true, y_pred)

        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Maximum F1-Score: {best_f1:.4f}\n")

    return best_thresh, best_f1

# 5. Run the sweep on validation predictions
best_threshold, max_f1 = find_best_threshold(y_val, y_val_proba)

# Evaluate the final tuned model setup
final_preds = (y_val_proba >= best_threshold).astype(int)
print("--- Final Classification Report ---")
print(classification_report(y_val, final_preds))


In [ ]:
X_test_tfidf = vectorizer.transform(test_df['question_text'])

test_probabilities = model.predict_proba(X_test_tfidf)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

submission_df = pd.DataFrame({
    'qid': test_df['qid'],
    'prediction': test_predictions
})

submission_df.to_csv('submission.csv', index=False)
